# Temporal Split Evaluation — N-1 Security Classifier

Offline experiment for the SiKDD paper. Compares random 80/20 split
against a temporal split (train Jan–Sep 2023, test Oct–Dec 2023)
to assess whether the Random Forest generalises to unseen time periods.

Dataset: `data/simulation_security_labels_n-1.csv`  
Feature set: `load_*`, `gen_*`, `sgen_*` only (7 post-simulation columns excluded)  
Model: `RandomForestClassifier(n_estimators=400, random_state=42)`

## 1. Load and inspect dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/simulation_security_labels_n-1.csv")

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

print(f"Shape:           {df.shape}")
print(f"Columns:         {len(df.columns)}")
print(f"Timestamp range: {df['timestamp'].min()} → {df['timestamp'].max()}")
print()
print("Class distribution:")
vc = df["status"].value_counts()
for label, count in vc.items():
    print(f"  {label:>10}: {count:>5}  ({count / len(df) * 100:.1f}%)")

Shape:           (8769, 273)
Columns:         273
Timestamp range: 2023-01-01 00:00:00 → 2023-12-31 23:00:00

Class distribution:
      secure:  4497  (51.3%)
    insecure:  4272  (48.7%)


## 2. Feature preparation

In [2]:
LABEL_COL = "status"

EXCLUDE_COLS = [
    "timestamp",
    "max_line_loading_percent_basecase",
    "min_bus_voltage_pu_basecase",
    "max_bus_voltage_pu_basecase",
    "max_line_loading_percent_contingency",
    "min_bus_voltage_pu_contingency",
    "max_bus_voltage_pu_contingency",
]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS and c != LABEL_COL]

print(f"n_features:             {len(feature_cols)}")
print(f"First 10 feature cols:  {feature_cols[:10]}")

n_features:             265
First 10 feature cols:  ['load_0_p_mw', 'load_1_p_mw', 'load_2_p_mw', 'load_3_p_mw', 'load_4_p_mw', 'load_5_p_mw', 'load_6_p_mw', 'load_7_p_mw', 'load_8_p_mw', 'load_9_p_mw']


## 3. Temporal split

In [3]:
df_sorted = df.sort_values("timestamp").reset_index(drop=True)

SPLIT_DATE = pd.Timestamp("2023-10-01 00:00:00")

df_train = df_sorted[df_sorted["timestamp"] < SPLIT_DATE]
df_test  = df_sorted[df_sorted["timestamp"] >= SPLIT_DATE]

print(f"n_train: {len(df_train)}")
print(f"n_test:  {len(df_test)}")
print(f"Train:   {df_train['timestamp'].min()} → {df_train['timestamp'].max()}")
print(f"Test:    {df_test['timestamp'].min()} → {df_test['timestamp'].max()}")
print()

for split_name, split_df in [("Train", df_train), ("Test", df_test)]:
    print(f"{split_name} class distribution:")
    vc = split_df["status"].value_counts()
    for label, count in vc.items():
        print(f"  {label:>10}: {count:>5}  ({count / len(split_df) * 100:.1f}%)")
    print()

n_train: 6561
n_test:  2208
Train:   2023-01-01 00:00:00 → 2023-09-30 23:00:00
Test:    2023-10-01 00:00:00 → 2023-12-31 23:00:00

Train class distribution:
      secure:  3424  (52.2%)
    insecure:  3137  (47.8%)

Test class distribution:
    insecure:  1135  (51.4%)
      secure:  1073  (48.6%)



## 4. Train Random Forest

In [4]:
from sklearn.ensemble import RandomForestClassifier

X_train_t = df_train[feature_cols].values
y_train_t = df_train[LABEL_COL].values
X_test_t  = df_test[feature_cols].values
y_test_t  = df_test[LABEL_COL].values

clf_temporal = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
clf_temporal.fit(X_train_t, y_train_t)
y_pred_t = clf_temporal.predict(X_test_t)
y_prob_t = clf_temporal.predict_proba(X_test_t)

print(f"Training complete. Classes: {clf_temporal.classes_}")

Training complete. Classes: ['insecure' 'secure']


## 5. Evaluate — all metrics

Metrics computed for both temporal split and random split (stratified 80/20).

In [5]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
)
from sklearn.model_selection import train_test_split


def compute_metrics(y_true, y_pred, y_prob, clf):
    insecure_idx = list(clf.classes_).index("insecure")
    y_prob_ins = y_prob[:, insecure_idx]
    y_bin = (y_true == "insecure").astype(int)
    return {
        "accuracy":           accuracy_score(y_true, y_pred),
        "f1_macro":           f1_score(y_true, y_pred, average="macro"),
        "precision_insecure": precision_score(y_true, y_pred, pos_label="insecure", zero_division=0),
        "recall_insecure":    recall_score(y_true, y_pred, pos_label="insecure", zero_division=0),
        "f1_insecure":        f1_score(y_true, y_pred, pos_label="insecure", zero_division=0),
        "roc_auc":            roc_auc_score(y_bin, y_prob_ins),
        "confusion_matrix":   confusion_matrix(y_true, y_pred, labels=["insecure", "secure"]),
    }


def print_metrics(m, title):
    cm = m["confusion_matrix"]
    print(f"=== {title} ===")
    print(f"  accuracy:            {m['accuracy']:.4f}")
    print(f"  f1_macro:            {m['f1_macro']:.4f}")
    print(f"  precision_insecure:  {m['precision_insecure']:.4f}")
    print(f"  recall_insecure:     {m['recall_insecure']:.4f}")
    print(f"  f1_insecure:         {m['f1_insecure']:.4f}")
    print(f"  roc_auc:             {m['roc_auc']:.4f}")
    print(f"  confusion_matrix (rows=true, cols=pred; order: insecure, secure):")
    print(f"    [[{cm[0,0]}, {cm[0,1]}], [{cm[1,0]}, {cm[1,1]}]]")
    print()


# --- Temporal split ---
m_temporal = compute_metrics(y_test_t, y_pred_t, y_prob_t, clf_temporal)
print_metrics(m_temporal, "Temporal split  (train Jan–Sep, test Oct–Dec 2023)")

# --- Random split ---
X_all = df_sorted[feature_cols].values
y_all = df_sorted[LABEL_COL].values

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
clf_random = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1)
clf_random.fit(X_tr_r, y_tr_r)
y_pred_r = clf_random.predict(X_te_r)
y_prob_r = clf_random.predict_proba(X_te_r)

m_random = compute_metrics(y_te_r, y_pred_r, y_prob_r, clf_random)
print_metrics(m_random, "Random split    (stratified 80/20, random_state=42)")

=== Temporal split  (train Jan–Sep, test Oct–Dec 2023) ===
  accuracy:            0.9094
  f1_macro:            0.9092
  precision_insecure:  0.9835
  recall_insecure:     0.8379
  f1_insecure:         0.9049
  roc_auc:             0.9783
  confusion_matrix (rows=true, cols=pred; order: insecure, secure):
    [[951, 184], [16, 1057]]

=== Random split    (stratified 80/20, random_state=42) ===
  accuracy:            0.9396
  f1_macro:            0.9395
  precision_insecure:  0.9390
  recall_insecure:     0.9368
  f1_insecure:         0.9379
  roc_auc:             0.9837
  confusion_matrix (rows=true, cols=pred; order: insecure, secure):
    [[800, 54], [52, 848]]



## 6. Comparison table

In [6]:
METRIC_KEYS = [
    ("accuracy",           "accuracy"),
    ("f1_macro",           "f1_macro"),
    ("precision_insecure", "precision_insecure"),
    ("recall_insecure",    "recall_insecure"),
    ("f1_insecure",        "f1_insecure"),
    ("roc_auc",            "roc_auc"),
]

col1 = "Metric"
col2 = "Random split"
col3 = "Temporal split"

print(f"{col1:<24} {col2:>14} {col3:>16}")
print("-" * 56)
for label, key in METRIC_KEYS:
    r = m_random[key]
    t = m_temporal[key]
    delta = t - r
    sign = "+" if delta >= 0 else ""
    print(f"{label:<24} {r:>14.3f} {t:>14.3f}   ({sign}{delta:.3f})")

Metric                     Random split   Temporal split
--------------------------------------------------------
accuracy                          0.940          0.909   (-0.030)
f1_macro                          0.940          0.909   (-0.030)
precision_insecure                0.939          0.983   (+0.044)
recall_insecure                   0.937          0.838   (-0.099)
f1_insecure                       0.938          0.905   (-0.033)
roc_auc                           0.984          0.978   (-0.005)


## 7. Brief interpretation

The temporal split (train Jan–Sep 2023, test Oct–Dec 2023) evaluates whether
the Random Forest generalises to operating conditions in unseen time periods,
which is more rigorous than a random split for time-series data.

**If temporal metrics ≈ random metrics (Δ < 0.02):** the model generalises
well across seasons, providing strong evidence for the SiKDD paper that the
learned `load_*`, `gen_*`, `sgen_*` feature patterns are stable over time.

**If temporal recall_insecure drops significantly vs. random split:** the model
misses more truly insecure states in Oct–Dec — the most dangerous failure mode
in N-1 security classification. This would need to be flagged in the paper and
addressed before deployment.

**Baseline for comparison (random split, 265 features):**  
accuracy=0.940, f1_macro=0.940, precision_insecure=0.935,
recall_insecure=0.943, f1_insecure=0.939, roc_auc=0.985.

> Fill in the temporal split results above and update this cell with
> 2–3 sentences summarising the actual findings for the paper.